# KPI Definition & Business Metric Design

This notebook demonstrates KPI computation, target validation, and revenue decomposition for HirePulse.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

BASE_DIR = Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from kpis.kpi_functions import (
    calculate_all_kpis,
    format_currency,
    format_percentage
)

DATA_FILE = BASE_DIR / "data" / "raw" / "customer_segment_data.csv"
TARGET_FILE = BASE_DIR / "kpis" / "kpi_validation_targets.json"

df = pd.read_csv(DATA_FILE)

print(f"Dataset: {DATA_FILE}")
print(f"Rows: {len(df)}")
print(f"Unique customers: {df['customer_id'].nunique()}")

df.head()


## KPI Calculations

The reusable KPI functions calculate six business metrics:

1. Average Revenue per Customer
2. Customer Churn Rate
3. Enterprise Revenue per Customer
4. SMB Revenue per Customer
5. Startup Revenue per Customer
6. Enterprise Revenue Share


In [ ]:
kpis = calculate_all_kpis(df)

for name, value in kpis.items():
    if "rate" in name or "share" in name:
        print(f"{name}: {format_percentage(value)}")
    else:
        print(f"{name}: {format_currency(value)}")


## KPI Target Validation

Each calculated KPI is compared against the defined minimum and maximum target range.

In [ ]:
with open(TARGET_FILE, "r", encoding="utf-8") as f:
    targets = json.load(f)

validation_rows = []

for name, value in kpis.items():
    target = targets[name]

    status = (
        "PASS"
        if target["min"] <= value <= target["max"]
        else "ALERT"
    )

    validation_rows.append({
        "kpi": name,
        "actual": value,
        "target_min": target["min"],
        "target_max": target["max"],
        "status": status
    })

validation_df = pd.DataFrame(validation_rows)

validation_df


## KPI Decomposition

Revenue is decomposed from total revenue to customer segment and then to product within each segment.

Each level is reconciled against the total revenue.

In [ ]:
total_revenue = df["revenue"].sum()

segment_revenue = (
    df.groupby("customer_type")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

product_revenue = (
    df.groupby(["customer_type", "product"])["revenue"]
    .sum()
)

print("Total Revenue:", format_currency(total_revenue))

print("\nRevenue by Segment:")
print(segment_revenue)

print("\nRevenue by Product within Segment:")
print(product_revenue)

print(
    "\nSegment reconciliation:",
    segment_revenue.sum() == total_revenue
)

print(
    "Product reconciliation:",
    product_revenue.sum() == total_revenue
)


## Customer Count Decomposition

In [ ]:
total_customers = df["customer_id"].nunique()

segment_customers = (
    df.groupby("customer_type")["customer_id"]
    .nunique()
)

print("Total Customers:", total_customers)
print("\nCustomers by Segment:")
print(segment_customers)

print(
    "\nCustomer reconciliation:",
    segment_customers.sum() == total_customers
)

revenue_per_customer = total_revenue / total_customers

print("\nRevenue per Customer:")
print(
    f"{format_currency(total_revenue)} / "
    f"{total_customers} = "
    f"{format_currency(revenue_per_customer)}"
)


## Summary

The notebook demonstrates:

- Reusable KPI calculation functions
- KPI target validation
- PASS/ALERT classification
- Revenue decomposition by customer segment
- Revenue decomposition by product
- Customer-count reconciliation
- Revenue-per-customer calculation
